# WO8c — Environment↔culture correspondence: political complexity (EA033)

Statistical-test notebook (no engine / API / UI). The trait is **EA033 jurisdictional hierarchy**
(ordinal, acephalous → 4-level state), the first genuinely speculative rung of the correspondence arc
— 8a and 8b were calibration on traits whose answers could be checked against intuition; 8c cannot be
pre-checked, so a null is a fully legitimate (and on the 8b base rate, modal) outcome.

**What this tests, and what it cannot.** The marginal complexity↔environment number will re-measure
EA033's empirical correlation with EA042 subsistence — uninformative alone. The question is the
**nested residual**, run two ways: net of {subsistence} and net of {subsistence + fixity} — because
fixity may be a mediator (environment → settlement → complexity), not a confound, so over-controlling
for it could remove real signal. The literature's environmental hypothesis (Carneiro circumscription:
states arise where arable land is bounded) is a relational spatial property the Climate-envelope
signature cannot express — **8c does not test circumscription**; a null here does not refute it. Part D
adds a cheap, physically distinct terrain (ruggedness) lens — a fragmentation proxy, opposite-signed
from circumscription's containment proxy — that narrows but does not close the terrain question.

**The effect-size floor (committed before this number is seen):** the 95th percentile of the
family-restricted permutation null R², read off the data, not picked by eye. Cross-checked against the
WO8b fixity residual (R²≈0.01–0.03, already judged uninterpretable) as an anchor.

Stats engine: `scripts/cdop/dbperm.py` (now also exposing the permutation-null R² distribution via
`return_null=True`, added for this WO's floor rule; `tests/cdop/test_dbperm.py` green, including the
new coverage). Substrate: extends `wo8b_substrate.parquet`. Terrain: `dplace.society_terrain`
(point-window grid relief, persisted for this WO — `scripts/cdop/persist_dplace_terrain.py`).

**Accept gate** is not "is it significant" — a defensible, reported effect size (marginal, nested×2,
ordinal, PERMDISP, terrain lens), interpretable whichever way it comes out. A null passes.

WO: `docs/cdop/pilot/wo8c_political complexity-EA033.md`.

In [2]:
# Cell 1
%matplotlib inline
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage

import scripts.shared.db_utils as db_utils
from scripts.cdop.dbperm import permanova, adonis_term, dbrda_trend, permdisp   # validated engine, now with return_null

ROOT = Path(db_utils.__file__).parent.parent.parent
CLDF = ROOT / 'data' / 'dplace' / 'cldf'
OUT  = ROOT / 'output' / 'cdop'


def z_euclid(df, cols):
    """z-scored Euclidean distance matrix over `cols` (complete-case; caller drops NaN)."""
    X = df[cols].to_numpy(float)
    Xz = (X - X.mean(0)) / X.std(0)
    G = Xz @ Xz.T
    d2 = np.diag(G)[:, None] + np.diag(G)[None, :] - 2 * G
    return np.sqrt(np.clip(d2, 0, None))


print(f"ready | dbperm loaded | wo8b substrate present: {(OUT / 'wo8b_substrate.parquet').exists()}")

ready | dbperm loaded | wo8b substrate present: True


In [3]:
# Cell 2 -- Part A.1: extend the WO8b substrate with EA033 jurisdictional hierarchy (ordinal 1..5).
# Sourced from the local CLDF (data/dplace/cldf), joined to the persisted substrate by soc_id.
sub = pd.read_parquet(OUT / 'wo8b_substrate.parquet')
codes = pd.read_csv(CLDF / 'codes.csv')
data  = pd.read_csv(CLDF / 'data.csv', low_memory=False)

# EA033 codes: use the codebook's own `ord` column (1..5); Missing data is ord=99 -> excluded.
c33 = codes[(codes['Var_ID'] == 'EA033') & (codes['ord'] < 90)].copy()
c33['ord'] = c33['ord'].astype(int)
ord33, lab33 = dict(zip(c33['ID'], c33['ord'])), dict(zip(c33['ID'], c33['Name']))

d33 = data.loc[data['Var_ID'] == 'EA033', ['Soc_ID', 'Code_ID']].copy()
d33 = d33[d33['Code_ID'].isin(ord33)]                       # keep real (non-missing) codes only
d33['complexity_ord']   = d33['Code_ID'].map(ord33).astype(int)
d33['complexity_label'] = d33['Code_ID'].map(lab33)
sub = sub.merge(d33[['Soc_ID', 'complexity_ord', 'complexity_label']],
                left_on='soc_id', right_on='Soc_ID', how='left').drop(columns='Soc_ID')

coded = sub['complexity_ord'].notna()
vc = (sub[coded].groupby(['complexity_ord', 'complexity_label']).size()
      .reset_index(name='n').sort_values('complexity_ord'))
print("\n".join([
    f"substrate societies (from wo8b, fixity+family already joined): {len(sub)}",
    f"EA033 complexity coded:  {int(coded.sum())} / {len(sub)}  (drop {int((~coded).sum())} uncoded)",
    "",
    "complexity gradient (ordinal : label : n):",
    vc.to_string(index=False),
]))

substrate societies (from wo8b, fixity+family already joined): 1133
EA033 complexity coded:  1012 / 1133  (drop 121 uncoded)

complexity gradient (ordinal : label : n):
 complexity_ord complexity_label   n
            1.0       Acephalous 464
            2.0        One level 295
            3.0       Two levels 137
            4.0     Three levels  75
            5.0      Four levels  41


In [4]:
# Cell 3 -- Part A.2: join the Part D terrain lens. Point-window (preferred, WO8c Part D) from
# `dplace.society_terrain` (persisted this WO via persist_dplace_terrain.py: 5x5, 1km-spaced grid,
# +-2km box, OpenTopoData mapzen batch -- no local DEM raster exists, checked before this WO); basin
# `smx-smn` (the WO's named fallback) joined alongside for comparison, via the hybas_id already in
# the substrate. relief_range_m = point-window max-min; landform_position = (mean-min)/(max-min),
# near 0 = valley-floor mass, near 0.5 = uniform slope.
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
conn = db_utils.db_connect()

terrain = pd.read_sql("""
    SELECT soc_id, grid_elev_min, grid_elev_max, grid_elev_mean,
           relief_range_m, landform_position, n_grid_resolved
    FROM dplace.society_terrain
""", conn)
sub = sub.merge(terrain, on='soc_id', how='left')

hybas_ids = tuple(int(h) for h in sub['hybas_id'].dropna().unique())
basin_elev = pd.read_sql(f"""
    SELECT hybas_id, ele_mt_smx, ele_mt_smn
    FROM basin08 WHERE hybas_id IN {hybas_ids}
""", conn)
conn.close()

# BasinATLAS NoData sentinel (-9999) must be masked before arithmetic (CLAUDE.md standing rule) --
# subtracting a raw sentinel would corrupt basin_relief_range_m for any nodata basin.
basin_elev[['ele_mt_smx', 'ele_mt_smn']] = basin_elev[['ele_mt_smx', 'ele_mt_smn']].replace(-9999, np.nan)
basin_elev['basin_relief_range_m'] = basin_elev['ele_mt_smx'] - basin_elev['ele_mt_smn']
sub = sub.merge(basin_elev[['hybas_id', 'basin_relief_range_m']], on='hybas_id', how='left')

pw_ok = sub['relief_range_m'].notna()
print("\n".join([
    f"point-window terrain resolved: {int(pw_ok.sum())} / {len(sub)}  "
    f"(all-25-grid-points cases: {int((sub['n_grid_resolved'] == 25).sum())})",
    f"basin smx-smn resolved (post NoData mask): {int(sub['basin_relief_range_m'].notna().sum())} / {len(sub)}",
    "",
    "point-window vs basin relief_range_m (m) -- summary, where both present:",
    sub.loc[pw_ok & sub['basin_relief_range_m'].notna(),
            ['relief_range_m', 'basin_relief_range_m']].describe().to_string(),
]))

point-window terrain resolved: 1133 / 1133  (all-25-grid-points cases: 1133)
basin smx-smn resolved (post NoData mask): 1133 / 1133

point-window vs basin relief_range_m (m) -- summary, where both present:
       relief_range_m  basin_relief_range_m
count     1133.000000           1133.000000
mean       211.232127            928.078553
std        284.409210            925.311745
min          0.000000              6.000000
25%         35.000000            214.000000
50%         83.000000            620.000000
75%        269.000000           1367.000000
max       1711.000000           7542.000000


In [5]:
# Cell 4 -- Part A.3: pre-test cell-count gate. Complexity is the *third* near-collinear cultural
# variable (with subsistence and fixity) -- report EA033 x subsistence, EA033 x fixity, EA033 x family
# BEFORE any test runs. Eligible n needs all four: complexity + subsistence + fixity + family.
elig = sub.dropna(subset=['complexity_ord', 'fixity_ord', 'family_id', 'ea042_subsistence']).copy()
elig['complexity_ord'] = elig['complexity_ord'].astype(int)

ct_subs = pd.crosstab(elig['complexity_label'], elig['ea042_subsistence'])
ct_fix  = pd.crosstab(elig['complexity_label'], elig['fixity_label'])
fam_sizes = elig.groupby('family_id').size()

sub.to_parquet(OUT / 'wo8c_substrate.parquet', index=False)

print("\n".join([
    f"eligible (complexity + fixity + family + subsistence all present): {len(elig)} / {len(sub)}",
    f"families: {int(elig['family_id'].nunique())}  |  singleton families: {int((fam_sizes == 1).sum())}  "
    f"|  societies in permutable (>=2) families: {int(fam_sizes[fam_sizes >= 2].sum())}",
    "",
    "complexity x subsistence -- scan for thin / empty cells:",
    ct_subs.to_string(),
    f"nonzero cells: min={int(ct_subs.values[ct_subs.values > 0].min())}  "
    f"median={int(np.median(ct_subs.values[ct_subs.values > 0]))}  "
    f"| empty cells={int((ct_subs.values == 0).sum())} of {ct_subs.size}",
    "",
    "complexity x fixity -- scan for thin / empty cells:",
    ct_fix.to_string(),
    f"nonzero cells: min={int(ct_fix.values[ct_fix.values > 0].min())}  "
    f"median={int(np.median(ct_fix.values[ct_fix.values > 0]))}  "
    f"| empty cells={int((ct_fix.values == 0).sum())} of {ct_fix.size}",
    "",
    f"saved -> {OUT / 'wo8c_substrate.parquet'}",
]))

eligible (complexity + fixity + family + subsistence all present): 891 / 1133
families: 78  |  singleton families: 13  |  societies in permutable (>=2) families: 878

complexity x subsistence -- scan for thin / empty cells:
ea042_subsistence  Extensive agriculture  Fishing  Gathering  Hunting  Intensive agriculture  Pastoralism
complexity_label                                                                                         
Acephalous                           149       59         76       50                     52            7
Four levels                            2        1          0        0                     36            0
One level                            139       11          6       20                     57           29
Three levels                          17        0          0        0                     43           10
Two levels                            54        0          0        0                     52           21
nonzero cells: min=1  median=36  |

In [6]:
# Cell 5 -- Part B: metric confirmation (drop-to-representative carried from 8b -- no re-litigation
# unless this sample's correlation structure diverges materially from 8a/8b's) + the declared 5->3
# complexity collapse (used only if the 5-level cells above are too thin for honest permutation).
PERM = 1999
an = pd.read_parquet(OUT / 'wo8c_substrate.parquet')
an = an.dropna(subset=['complexity_ord', 'fixity_ord', 'family_id', 'ea042_subsistence']).reset_index(drop=True)
an['complexity_ord'] = an['complexity_ord'].astype(int)
an['ari_log'] = np.log1p(an['ari_ix_sav'])
fam = an['family_id'].to_numpy()

# fixity4 is NOT a persisted substrate column -- WO8b computed it in-notebook from fixity_ord and
# never wrote it back to the parquet (its Cell 4 saved `sub` before this line ran). Recompute here
# with WO8b's own declared 8->4 collapse (identical mapping, not re-derived).
FIXITY_COLLAPSE = {1: 'mobile', 2: 'mobile', 3: 'semi', 4: 'semi',
                   5: 'sedentary', 6: 'sedentary', 7: 'sedentary', 8: 'complex'}
an['fixity4'] = an['fixity_ord'].astype(int).map(FIXITY_COLLAPSE)

KEEP5 = ['ari_log', 'pre_mm_syr', 'run_mm_syr', 'temperature_annual', 'tmp_seas_amp']
REP3  = ['ari_log', 'temperature_annual', 'tmp_seas_amp']   # carried from WO8b

corr = an[KEEP5].corr()
water_r = corr.loc[['ari_log', 'pre_mm_syr', 'run_mm_syr'], ['ari_log', 'pre_mm_syr', 'run_mm_syr']]
thermal_r = corr.loc['temperature_annual', 'tmp_seas_amp']

# Declared 5->3 collapse -- a common trichotomy in the political-evolution literature (acephalous /
# intermediate polities / state-level), not fit to this sample's result.
COLLAPSE3 = {1: 'acephalous', 2: 'intermediate', 3: 'intermediate', 4: 'state', 5: 'state'}
an['complexity3'] = an['complexity_ord'].map(COLLAPSE3)

lines = [f"analysis set (Part C universe): {len(an)} societies",
         f"fixity4 recomputed from fixity_ord: " + an['fixity4'].value_counts().to_dict().__repr__(),
         "",
         "water-block correlations (on THIS sample -- compare to WO8a's 0.66-0.83):",
         water_r.round(3).to_string(),
         f"thermal pair (temperature_annual, tmp_seas_amp) r = {thermal_r:.3f}  (compare to WO8a's -0.83)",
         "",
         "Carried forward: drop-to-representative (REP3) unless the block above diverges materially",
         "from WO8a/WO8b's structure.",
         "",
         "declared complexity3 collapse counts (used only if the 5-level cells are too thin):",
         an['complexity3'].value_counts().reindex(['acephalous', 'intermediate', 'state']).to_string()]
print("\n".join(lines))

analysis set (Part C universe): 891 societies
fixity4 recomputed from fixity_ord: {'sedentary': 569, 'mobile': 220, 'semi': 79, 'complex': 23}

water-block correlations (on THIS sample -- compare to WO8a's 0.66-0.83):
            ari_log  pre_mm_syr  run_mm_syr
ari_log       1.000       0.758       0.647
pre_mm_syr    0.758       1.000       0.826
run_mm_syr    0.647       0.826       1.000
thermal pair (temperature_annual, tmp_seas_amp) r = -0.837  (compare to WO8a's -0.83)

Carried forward: drop-to-representative (REP3) unless the block above diverges materially
from WO8a/WO8b's structure.

declared complexity3 collapse counts (used only if the 5-level cells are too thin):
complexity3
acephalous      393
intermediate    389
state           109


In [7]:
# Cell 6 -- Part C marginal: does complexity track environment? Expected strong, expected mostly
# subsistence -- reported as the confound baseline, not a finding (WO's own framing).
# factor (complexity3) + ordinal trend (complexity_ord, full 1-5) + PERMDISP, family-restricted.
D = z_euclid(an, REP3)
mf = permanova(D, an['complexity3'].to_numpy(), blocks=fam, n_perm=PERM, seed=1)
mo = dbrda_trend(D, an['complexity_ord'].to_numpy().astype(float), blocks=fam, n_perm=PERM, seed=1)
md = permdisp(D, an['complexity3'].to_numpy(), n_perm=PERM, seed=1)
print("\n".join([
    "MARGINAL (complexity vs Climate-envelope distance, drop-to-representative, family-restricted):",
    f"  factor (3-level)  : R2={mf.R2:.4f}  F={mf.F:.2f}  p={mf.p:.4f}  (df {mf.df1},{mf.df2})",
    f"  ordinal trend (1-5): R2={mo.R2:.4f}  F={mo.F:.2f}  p={mo.p:.4f}  (1-df monotonic)",
    f"  PERMDISP           : F={md.F:.2f}  p={md.p:.4f}  "
    + ("[dispersion differs -- read a location shift with care]" if md.p < 0.05
       else "[dispersions homogeneous -- a location shift is real, not a spread artifact]"),
    f"  group dispersions: " + ", ".join(f"{k}={v:.2f}" for k, v in md.group_mean_dist.items()),
    "",
    "Expected strong and mostly subsistence-in-disguise (WO framing) -- this number is the confound",
    "baseline, not the finding. The nested residual (Cell 7-8) is the actual test.",
]))

MARGINAL (complexity vs Climate-envelope distance, drop-to-representative, family-restricted):
  factor (3-level)  : R2=0.0500  F=23.36  p=0.0680  (df 2,888)
  ordinal trend (1-5): R2=0.0056  F=5.01  p=0.3815  (1-df monotonic)
  PERMDISP           : F=40.78  p=0.0005  [dispersion differs -- read a location shift with care]
  group dispersions: state=1.57, acephalous=1.72, intermediate=1.23

Expected strong and mostly subsistence-in-disguise (WO framing) -- this number is the confound
baseline, not the finding. The nested residual (Cell 7-8) is the actual test.


In [8]:
# Cell 7 -- Part C nested SPEC 1: complexity NET OF subsistence only (Freedman-Lane, within family).
# Subsistence is the cleaner confound (economy as common cause of both complexity and environment).
subs_arr = an['ea042_subsistence'].to_numpy()
n1f = adonis_term(D, an['complexity3'].to_numpy(), covars=subs_arr, blocks=fam, n_perm=PERM, seed=1)
n1o = adonis_term(D, an['complexity_ord'].to_numpy().astype(float), covars=subs_arr, term_ordinal=True,
                  blocks=fam, n_perm=PERM, seed=1)
print("\n".join([
    "NESTED SPEC 1 (complexity | subsistence, Freedman-Lane within family):",
    f"  factor (3-level)  : R2={n1f.R2:.4f}  F={n1f.F:.2f}  p={n1f.p:.4f}   (marginal was {mf.R2:.4f})",
    f"  ordinal trend      : R2={n1o.R2:.4f}  F={n1o.F:.2f}  p={n1o.p:.4f}   (marginal was {mo.R2:.4f})",
    f"  marginal - nested1 R2 gap (factor):   {mf.R2 - n1f.R2:.4f}",
    f"  marginal - nested1 R2 gap (ordinal):  {mo.R2 - n1o.R2:.4f}",
]))

NESTED SPEC 1 (complexity | subsistence, Freedman-Lane within family):
  factor (3-level)  : R2=0.0169  F=12.43  p=0.0320   (marginal was 0.0500)
  ordinal trend      : R2=0.0085  F=12.33  p=0.2270   (marginal was 0.0056)
  marginal - nested1 R2 gap (factor):   0.0331
  marginal - nested1 R2 gap (ordinal):  -0.0029


In [9]:
# Cell 8 -- Part C nested SPEC 2: complexity NET OF subsistence + fixity (Freedman-Lane, within
# family, TWO covariates via adonis_term's covars=list). Fixity may be a MEDIATOR (environment ->
# settlement -> complexity), not a confound -- controlling for it can remove real signal, not just
# noise. The gap between SPEC 1 and SPEC 2 measures how much fixity absorbs: confound-control if the
# gap is small, signal-eating if large. Report both numbers side by side; never headline SPEC 2 alone.
fix_arr = an['fixity4'].to_numpy()   # WO8b's declared 8->4 collapse, already in the substrate
n2f = adonis_term(D, an['complexity3'].to_numpy(), covars=[subs_arr, fix_arr],
                  blocks=fam, n_perm=PERM, seed=1)
n2o = adonis_term(D, an['complexity_ord'].to_numpy().astype(float), covars=[subs_arr, fix_arr],
                  term_ordinal=True, blocks=fam, n_perm=PERM, seed=1)
print("\n".join([
    "NESTED SPEC 2 (complexity | subsistence + fixity, Freedman-Lane within family):",
    f"  factor (3-level)  : R2={n2f.R2:.4f}  F={n2f.F:.2f}  p={n2f.p:.4f}",
    f"  ordinal trend      : R2={n2o.R2:.4f}  F={n2o.F:.2f}  p={n2o.p:.4f}",
    "",
    "SPEC 1 vs SPEC 2 -- side by side (never headline SPEC 2 without SPEC 1 beside it):",
    f"  factor:   spec1 R2={n1f.R2:.4f}   spec2 R2={n2f.R2:.4f}   "
    f"fixity absorbs {n1f.R2 - n2f.R2:+.4f}",
    f"  ordinal:  spec1 R2={n1o.R2:.4f}   spec2 R2={n2o.R2:.4f}   "
    f"fixity absorbs {n1o.R2 - n2o.R2:+.4f}",
    "",
    "COLLINEARITY CAVEAT: complexity, subsistence, and fixity are three near-collinear cultural",
    "variables (worse here than WO8b's two-variable case) -- whatever residual survives both specs",
    "rests on whatever independent complexity-variation remains after holding the other two; if that",
    "is thin, the residual is an overlap artifact, not a fact about nature (WO8c Part A/C proviso).",
]))

NESTED SPEC 2 (complexity | subsistence + fixity, Freedman-Lane within family):
  factor (3-level)  : R2=0.0181  F=14.07  p=0.0095
  ordinal trend      : R2=0.0102  F=15.63  p=0.0210

SPEC 1 vs SPEC 2 -- side by side (never headline SPEC 2 without SPEC 1 beside it):
  factor:   spec1 R2=0.0169   spec2 R2=0.0181   fixity absorbs -0.0011
  ordinal:  spec1 R2=0.0085   spec2 R2=0.0102   fixity absorbs -0.0017

COLLINEARITY CAVEAT: complexity, subsistence, and fixity are three near-collinear cultural
variables (worse here than WO8b's two-variable case) -- whatever residual survives both specs
rests on whatever independent complexity-variation remains after holding the other two; if that
is thin, the residual is an overlap artifact, not a fact about nature (WO8c Part A/C proviso).


In [10]:
# Cell 9 -- THE EFFECT-SIZE FLOOR (committed rule, WO8c precondition): floor = 95th percentile of the
# family-restricted permutation-null R2 distribution -- read off the data, not picked by eye. Applied
# to both nested specs, factor and ordinal. Same seed/n_perm as Cells 7-8, so the observed R2 here is
# identical to theirs; return_null=True additionally returns the null array (dbperm.py, this WO).
_, null_n1f = adonis_term(D, an['complexity3'].to_numpy(), covars=subs_arr,
                          blocks=fam, n_perm=PERM, seed=1, return_null=True)
_, null_n1o = adonis_term(D, an['complexity_ord'].to_numpy().astype(float), covars=subs_arr,
                          term_ordinal=True, blocks=fam, n_perm=PERM, seed=1, return_null=True)
_, null_n2f = adonis_term(D, an['complexity3'].to_numpy(), covars=[subs_arr, fix_arr],
                          blocks=fam, n_perm=PERM, seed=1, return_null=True)
_, null_n2o = adonis_term(D, an['complexity_ord'].to_numpy().astype(float), covars=[subs_arr, fix_arr],
                          term_ordinal=True, blocks=fam, n_perm=PERM, seed=1, return_null=True)

# WO8b's own fixity residual (nested | subsistence), already judged "no interpretable independent
# effect" -- the cross-check anchor named in the WO. Hardcoded from wo8b_findings.md / the wo8b
# notebook's Cell 7 output (not re-run here; it is the prior WO's closed result).
WO8B_FIXITY_RESIDUAL = {'factor': 0.0334, 'ordinal': 0.0108}

def _verdict(label, R2, null_dist, anchor=None):
    floor = np.percentile(null_dist, 95)
    verdict = "INTERPRETABLE" if R2 > floor else "no interpretable independent effect (sub-floor)"
    line = f"  {label:34}R2={R2:.4f}   floor(95th pct null)={floor:.4f}   -> {verdict}"
    if anchor is not None:
        line += f"   [WO8b fixity anchor: {anchor:.4f}]"
    return line, floor

lines = ["EFFECT-SIZE FLOOR (95th percentile of family-restricted permutation-null R2):"]
l, f1 = _verdict("nested SPEC1 (|subsistence) factor", n1f.R2, null_n1f, WO8B_FIXITY_RESIDUAL['factor'])
lines.append(l)
l, f2 = _verdict("nested SPEC1 (|subsistence) ordinal", n1o.R2, null_n1o, WO8B_FIXITY_RESIDUAL['ordinal'])
lines.append(l)
l, f3 = _verdict("nested SPEC2 (|subs+fixity) factor", n2f.R2, null_n2f, WO8B_FIXITY_RESIDUAL['factor'])
lines.append(l)
l, f4 = _verdict("nested SPEC2 (|subs+fixity) ordinal", n2o.R2, null_n2o, WO8B_FIXITY_RESIDUAL['ordinal'])
lines.append(l)
lines += ["",
    f"Null-floor range across the four tests: {min(f1,f2,f3,f4):.4f} - {max(f1,f2,f3,f4):.4f}",
    f"WO8b fixity-residual anchor: factor={WO8B_FIXITY_RESIDUAL['factor']:.4f}  "
    f"ordinal={WO8B_FIXITY_RESIDUAL['ordinal']:.4f}",
    "If the null-floor and the fixity anchor disagree materially, that gap is itself informative",
    "about how much n~1000 inflates small effects here (WO8c precondition) -- report it, don't pick one.",
]
print("\n".join(lines))

EFFECT-SIZE FLOOR (95th percentile of family-restricted permutation-null R2):
  nested SPEC1 (|subsistence) factorR2=0.0169   floor(95th pct null)=0.0157   -> INTERPRETABLE   [WO8b fixity anchor: 0.0334]
  nested SPEC1 (|subsistence) ordinalR2=0.0085   floor(95th pct null)=0.0100   -> no interpretable independent effect (sub-floor)   [WO8b fixity anchor: 0.0108]
  nested SPEC2 (|subs+fixity) factorR2=0.0181   floor(95th pct null)=0.0153   -> INTERPRETABLE   [WO8b fixity anchor: 0.0334]
  nested SPEC2 (|subs+fixity) ordinalR2=0.0102   floor(95th pct null)=0.0090   -> INTERPRETABLE   [WO8b fixity anchor: 0.0108]

Null-floor range across the four tests: 0.0090 - 0.0157
WO8b fixity-residual anchor: factor=0.0334  ordinal=0.0108
If the null-floor and the fixity anchor disagree materially, that gap is itself informative
about how much n~1000 inflates small effects here (WO8c precondition) -- report it, don't pick one.


In [12]:
# Cell 10 -- Part D: the cheap terrain lens. Tests RUGGEDNESS (fragmentation proxy), not ENCLOSURE
# (containment proxy, circumscription's actual variable, unbuilt) -- opposite expected signs, so this
# lens has NO clean directional prediction. A null narrows the terrain question but does not close it.
# Point-window (relief_range_m, landform_position), preferred per WO8c Part D; basin smx-smn is the
# named fallback (checked below, not expected to be needed -- point-window resolved 1133/1133 when
# persist_dplace_terrain.py ran).
terr_an = an.dropna(subset=['relief_range_m', 'landform_position']).reset_index(drop=True)
n_missing_pw = len(an) - len(terr_an)
print(f"point-window terrain available: {len(terr_an)} / {len(an)} of the Part C analysis set "
      f"({n_missing_pw} missing -> basin smx-smn fallback would apply here if nonzero)")

TERRAIN = ['relief_range_m', 'landform_position']
Dt = z_euclid(terr_an, TERRAIN)
fam_t = terr_an['family_id'].to_numpy()

tf = permanova(Dt, terr_an['complexity3'].to_numpy(), blocks=fam_t, n_perm=PERM, seed=1)
to = dbrda_trend(Dt, terr_an['complexity_ord'].to_numpy().astype(float), blocks=fam_t, n_perm=PERM, seed=1)
td = permdisp(Dt, terr_an['complexity3'].to_numpy(), n_perm=PERM, seed=1)

# Floor applied to BOTH factor and ordinal -- Cell 10's first draft only computed it for the factor
# test, leaving the ordinal's borderline p=0.049 unadjudicated by the same rule everything else in
# this notebook is held to. Fixed before reading anything into that number.
_, null_tf = permanova(Dt, terr_an['complexity3'].to_numpy(), blocks=fam_t, n_perm=PERM, seed=1, return_null=True)
_, null_to = dbrda_trend(Dt, terr_an['complexity_ord'].to_numpy().astype(float), blocks=fam_t,
                         n_perm=PERM, seed=1, return_null=True)
floor_tf = np.percentile(null_tf, 95)
floor_to = np.percentile(null_to, 95)

print("\n".join([
    "",
    "TERRAIN LENS (ruggedness/landform-position vs complexity, family-restricted, marginal --",
    "no subsistence/fixity covariates: terrain is a DIFFERENT physical question, not folded into",
    "the Climate-envelope distance):",
    f"  factor (3-level)  : R2={tf.R2:.4f}  F={tf.F:.2f}  p={tf.p:.4f}   "
    f"floor(95th pct null)={floor_tf:.4f}   "
    f"-> {'INTERPRETABLE' if tf.R2 > floor_tf else 'no interpretable independent effect (sub-floor)'}",
    f"  ordinal trend      : R2={to.R2:.4f}  F={to.F:.2f}  p={to.p:.4f}   "
    f"floor(95th pct null)={floor_to:.4f}   "
    f"-> {'INTERPRETABLE' if to.R2 > floor_to else 'no interpretable independent effect (sub-floor)'}",
    f"  PERMDISP           : F={td.F:.2f}  p={td.p:.4f}  "
    + ("[dispersion differs]" if td.p < 0.05 else "[dispersions homogeneous]"),
    f"  group dispersions: " + ", ".join(f"{k}={v:.2f}" for k, v in td.group_mean_dist.items()),
    "",
    "Reading: a positive here says rough/broken country associates with complexity level (sign not",
    "predicted -- could run either way per the WO); a null says ruggedness is narrowed as a channel,",
    "not excluded (enclosure -- containment, not fragmentation -- remains untested; Forward register).",
]))


TERRAIN LENS (ruggedness/landform-position vs complexity, family-restricted, marginal --
no subsistence/fixity covariates: terrain is a DIFFERENT physical question, not folded into
the Climate-envelope distance):
  factor (3-level)  : R2=0.0119  F=5.31  p=0.2390   floor(95th pct null)=0.0151   -> no interpretable independent effect (sub-floor)
  ordinal trend      : R2=0.0078  F=6.93  p=0.0490   floor(95th pct null)=0.0077   -> INTERPRETABLE
  PERMDISP           : F=5.33  p=0.0300  [dispersion differs]
  group dispersions: state=1.11, acephalous=1.28, intermediate=1.11

Reading: a positive here says rough/broken country associates with complexity level (sign not
predicted -- could run either way per the WO); a null says ruggedness is narrowed as a channel,
not excluded (enclosure -- containment, not fragmentation -- remains untested; Forward register).


In [13]:
# Cell 11 -- STABILITY CHECK on the floor verdicts. Cell 9/10's floors are 95th-percentile estimates
# from a finite permutation sample (n_perm=1999) and carry their own Monte Carlo noise; several of
# the Part C/D clearances were within ~10-20% of their floor, which is inside the range a different
# seed could plausibly flip. Rerun at a different seed AND a much larger n_perm (9999, sharper
# percentile estimate) for every nested/terrain factor+ordinal test; report whether each verdict holds.
PERM2 = 9999
SEED2 = 7

checks = [
    ("SPEC1 factor",   n1f.R2, D,  an['complexity3'].to_numpy(),                    subs_arr,          False, fam),
    ("SPEC1 ordinal",  n1o.R2, D,  an['complexity_ord'].to_numpy().astype(float),   subs_arr,          True,  fam),
    ("SPEC2 factor",   n2f.R2, D,  an['complexity3'].to_numpy(),                    [subs_arr,fix_arr], False, fam),
    ("SPEC2 ordinal",  n2o.R2, D,  an['complexity_ord'].to_numpy().astype(float),   [subs_arr,fix_arr], True,  fam),
    ("terrain factor", tf.R2,  Dt, terr_an['complexity3'].to_numpy(),               None,              False, fam_t),
    ("terrain ordinal",to.R2,  Dt, terr_an['complexity_ord'].to_numpy().astype(float), None,           True,  fam_t),
]
orig_floors = {"SPEC1 factor": f1, "SPEC1 ordinal": f2, "SPEC2 factor": f3, "SPEC2 ordinal": f4,
              "terrain factor": floor_tf, "terrain ordinal": floor_to}

lines = [f"STABILITY CHECK -- seed={SEED2}, n_perm={PERM2} (vs original seed=1, n_perm={PERM}):",
         f"  {'test':18}{'R2':>8}{'orig floor':>12}{'new floor':>12}  {'orig':<13}{'new':<13}same?"]
any_flip = False
for label, R2, Dm, term, covars, ordv, fam_arr in checks:
    if covars is None:
        _, null2 = permanova(Dm, term, blocks=fam_arr, n_perm=PERM2, seed=SEED2, return_null=True)
    else:
        _, null2 = adonis_term(Dm, term, covars=covars, term_ordinal=ordv, blocks=fam_arr,
                               n_perm=PERM2, seed=SEED2, return_null=True)
    orig_floor = orig_floors[label]
    new_floor = np.percentile(null2, 95)
    orig_v = "INTERP" if R2 > orig_floor else "sub-floor"
    new_v = "INTERP" if R2 > new_floor else "sub-floor"
    flip = orig_v != new_v
    any_flip = any_flip or flip
    lines.append(f"  {label:18}{R2:>8.4f}{orig_floor:>12.4f}{new_floor:>12.4f}  {orig_v:<13}{new_v:<13}"
                 + ("<<< FLIPPED" if flip else "same"))

lines.append("")
lines.append("ANY VERDICT FLIPPED: " + ("YES -- treat the flipped test(s) as unresolved, not settled."
             if any_flip else "no -- all six verdicts hold under a different seed and 5x the permutations."))
print("\n".join(lines))

STABILITY CHECK -- seed=7, n_perm=9999 (vs original seed=1, n_perm=1999):
  test                    R2  orig floor   new floor  orig         new          same?
  SPEC1 factor        0.0169      0.0157      0.0156  INTERP       INTERP       same
  SPEC1 ordinal       0.0085      0.0100      0.0100  sub-floor    sub-floor    same
  SPEC2 factor        0.0181      0.0153      0.0153  INTERP       INTERP       same
  SPEC2 ordinal       0.0102      0.0090      0.0091  INTERP       INTERP       same
  terrain factor      0.0119      0.0151      0.0153  sub-floor    sub-floor    same
  terrain ordinal     0.0078      0.0077      0.0079  INTERP       sub-floor    <<< FLIPPED

ANY VERDICT FLIPPED: YES -- treat the flipped test(s) as unresolved, not settled.


In [14]:
# Cell 12 -- APPENDIX: retroactive floor check on WO8b's fixity residual. WO8b's "no interpretable
# independent effect" call (R2=0.0334 factor / 0.0108 ordinal, nested | subsistence) was a judgment
# call made before return_null existed -- it was never actually checked against ITS OWN permutation-
# null floor, only cited here as a fixed anchor. WO8c's SPEC2 ordinal residual (0.0102, ruled
# interpretable) is numerically almost identical to WO8b's fixity ordinal residual (0.0108, ruled
# uninterpretable) -- an inconsistency worth resolving before either number is used as a comparison
# point. Reconstructs WO8b's EXACT design (its own 918-society universe, not WO8c's 891; same REP3
# metric, same fixity4 collapse, same seed/n_perm) from the untouched wo8b_substrate.parquet -- this
# does not modify the frozen wo8b notebook, it just adds return_null to the identical computation.
an8b = pd.read_parquet(OUT / 'wo8b_substrate.parquet')
an8b = an8b.dropna(subset=['fixity_ord', 'family_id', 'ea042_subsistence']).reset_index(drop=True)
an8b['fixity_ord'] = an8b['fixity_ord'].astype(int)
an8b['ari_log'] = np.log1p(an8b['ari_ix_sav'])
fam8b = an8b['family_id'].to_numpy()

FIXITY_COLLAPSE = {1: 'mobile', 2: 'mobile', 3: 'semi', 4: 'semi',
                   5: 'sedentary', 6: 'sedentary', 7: 'sedentary', 8: 'complex'}
an8b['fixity4'] = an8b['fixity_ord'].map(FIXITY_COLLAPSE)
subs8b = an8b['ea042_subsistence'].to_numpy()

D8b = z_euclid(an8b, REP3)   # REP3 unchanged from WO8b: ari_log, temperature_annual, tmp_seas_amp

n1f_8b, null_8b_f = adonis_term(D8b, an8b['fixity4'].to_numpy(), covars=subs8b,
                                blocks=fam8b, n_perm=1999, seed=1, return_null=True)
n1o_8b, null_8b_o = adonis_term(D8b, an8b['fixity_ord'].to_numpy().astype(float), covars=subs8b,
                                term_ordinal=True, blocks=fam8b, n_perm=1999, seed=1, return_null=True)

floor_8b_f = np.percentile(null_8b_f, 95)
floor_8b_o = np.percentile(null_8b_o, 95)

print("\n".join([
    f"WO8b reconstruction: {len(an8b)} societies (matches WO8b's own reported 918)",
    "",
    f"fidelity check -- these R2 must match WO8b's reported 0.0334 / 0.0108 exactly (same design,",
    f"seed, n_perm; only difference is requesting return_null this time):",
    f"  factor : R2={n1f_8b.R2:.4f}  (WO8b reported 0.0334)",
    f"  ordinal: R2={n1o_8b.R2:.4f}  (WO8b reported 0.0108)",
    "",
    "RETROACTIVE FLOOR CHECK -- WO8b's own residual against WO8b's own permutation-null floor:",
    f"  factor : R2={n1f_8b.R2:.4f}   floor(95th pct null)={floor_8b_f:.4f}   -> "
    f"{'INTERPRETABLE' if n1f_8b.R2 > floor_8b_f else 'no interpretable independent effect (sub-floor)'}",
    f"  ordinal: R2={n1o_8b.R2:.4f}   floor(95th pct null)={floor_8b_o:.4f}   -> "
    f"{'INTERPRETABLE' if n1o_8b.R2 > floor_8b_o else 'no interpretable independent effect (sub-floor)'}",
    "",
    "If this disagrees with WO8b's original judgment-call verdict ('no interpretable independent",
    "effect' for both), WO8b's closed finding needs a formal amendment note, not a silent overwrite.",
]))

WO8b reconstruction: 918 societies (matches WO8b's own reported 918)

fidelity check -- these R2 must match WO8b's reported 0.0334 / 0.0108 exactly (same design,
seed, n_perm; only difference is requesting return_null this time):
  factor : R2=0.0334  (WO8b reported 0.0334)
  ordinal: R2=0.0108  (WO8b reported 0.0108)

RETROACTIVE FLOOR CHECK -- WO8b's own residual against WO8b's own permutation-null floor:
  factor : R2=0.0334   floor(95th pct null)=0.0104   -> INTERPRETABLE
  ordinal: R2=0.0108   floor(95th pct null)=0.0030   -> INTERPRETABLE

If this disagrees with WO8b's original judgment-call verdict ('no interpretable independent
effect' for both), WO8b's closed finding needs a formal amendment note, not a silent overwrite.


## Accept gate — pending Karl's run

Not filled in — this section gets written from actual Cell 1–10 output after Karl runs the notebook
cell by cell and reports back, not before (per the arc's own process lesson from WO8b: no number is
stated as a finding until it has been read off real output).

Checklist the gate requires (WO8c § Accept gate): pre-test cell counts (Cell 4); the residual read
against the committed floor, 95th-percentile permutation-null R² (Cell 9), fixity-residual cross-check
alongside; the collinearity caveat stated with the confound-share (Cell 8); SPEC 1 and SPEC 2 numbers
reported side by side, never SPEC 2 alone (Cell 8); the terrain-lens result, interpretable whichever way
it came out (Cell 10); and an explicit statement of what a null does and does not rule out (it does not
rule out circumscription, which is untested — Forward register, `wo8c_political complexity-EA033.md`).